

*   harmony (two or more notes sounding together to build chords and add depth),
*   melody (a sequence of pitches and rhythms that form a recognizable tune),
*   rhythm (the pattern and placement of sounds in time).





Cell 1 — Setup & Helper Functions

In [47]:
import numpy as np
from IPython.display import Audio, display

SR = 44100  # Sample rate (samples per second)

# Equal temperament: MIDI note -> frequency in Hz
def note_to_freq(midi_note):
    return 440.0 * (2 ** ((midi_note - 69) / 12))

# Generate a sine tone with an ADSR envelope (so notes don't click)
def tone(freq, duration, amp=0.5, sr=SR):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    wave = np.sin(2 * np.pi * freq * t)

    # Attack-Decay-Sustain-Release envelope
    n = len(wave)
    a, d, r = int(0.01 * sr), int(0.1 * sr), int(0.2 * sr)
    s = max(n - a - d - r, 0)
    env = np.concatenate([
        np.linspace(0, 1, a),
        np.linspace(1, 0.7, d),
        np.full(s, 0.7),
        np.linspace(0.7, 0, r)
    ])[:n]
    return amp * wave * env

# Mix a list of (freq, duration) events sequentially
def sequence(events, amp=0.4):
    return np.concatenate([tone(f, d, amp) for f, d in events])

# Mix several voices together (same length), normalized
def mix(voices):
    length = max(len(v) for v in voices)
    out = np.zeros(length)
    for v in voices:
        out[:len(v)] += v
    return out / max(1, np.max(np.abs(out)))

Cell 2 — 🎼 Harmony (chords: notes sounding together)

In [48]:
# A major triad: C4 (60), E4 (64), G4 (67)
chord_notes = [60, 64, 67]
chord = mix([tone(note_to_freq(n), 1.5, amp=0.4) for n in chord_notes])
print("🎼 Harmony — C major triad (C-E-G played simultaneously)")
display(Audio(chord, rate=SR))

# Try a chord progression: C -> Am -> F -> G
progression = [
    [60, 64, 67],  # C major
    [57, 60, 64],  # A minor
    [53, 57, 60],  # F major
    [55, 59, 62],  # G major
]
prog_audio = np.concatenate([
    mix([tone(note_to_freq(n), 0.8, amp=0.4) for n in chord])
    for chord in progression
])
print("🎼 Harmony — I–vi–IV–V progression")
display(Audio(prog_audio, rate=SR))

🎼 Harmony — C major triad (C-E-G played simultaneously)


🎼 Harmony — I–vi–IV–V progression


Cell 3 — 🎶 Melody (pitches + rhythm forming a tune)

In [49]:
# # Simple melody: (MIDI note, duration in seconds)
# # "Twinkle Twinkle Little Star" opening phrase
# melody = [
#     (60, 0.5), (60, 0.5), (67, 0.5), (67, 0.5),
#     (69, 0.5), (69, 0.5), (67, 1.0),
#     (65, 0.5), (65, 0.5), (64, 0.5), (64, 0.5),
#     (62, 0.5), (62, 0.5), (60, 1.0),
# ]

# events = [(note_to_freq(n), d) for n, d in melody]
# melody_audio = sequence(events, amp=0.4)

# print("🎶 Melody — Twinkle Twinkle Little Star")
# display(Audio(melody_audio, rate=SR))

In [50]:
import numpy as np
from IPython.display import Audio, display

SR = 44100

def note_to_freq(midi_note):
    """Converts MIDI note to frequency in Hz."""
    return 440.0 * (2.0 ** ((midi_note - 69) / 12.0))

def electric_guitar_note(freq, duration, amp=0.6, drive=3.5, sr=SR):
    """
    Synthesizes a single electric guitar note through an overdriven tube amp
    and 12-inch speaker cabinet model.
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    n = len(t)
    if n == 0:
        return np.array([])

    # 1. Delayed finger vibrato (blues/rock style, blooms after ~150ms)
    vib_rate = 5.6  # Hz
    vib_depth = 0.012
    vib_ramp = np.clip((t - 0.15) / 0.20, 0.0, 1.0)
    inst_freq = freq * (1.0 + vib_depth * vib_ramp * np.sin(2.0 * np.pi * vib_rate * t))
    phase = 2.0 * np.pi * np.cumsum(inst_freq) / sr

    # 2. Steel string harmonic series (pickup position blend)
    # Higher harmonics decay faster than lower harmonics
    harmonics = [
        (1, 1.00, 1.5),   # Fundamental (strong sustain)
        (2, 0.75, 2.2),   # 2nd harmonic
        (3, 0.50, 3.0),   # 3rd harmonic
        (4, 0.35, 4.0),   # 4th harmonic
        (5, 0.20, 5.0),   # 5th harmonic
        (6, 0.12, 6.5),   # 6th harmonic
    ]

    string_sound = np.zeros_like(t)
    for mult, h_amp, decay_rate in harmonics:
        env = np.exp(-decay_rate * t)
        string_sound += h_amp * env * np.sin(mult * phase)

    # 3. Plectrum / Pick attack transient (first 10 ms click)
    pick_noise = np.random.uniform(-1.0, 1.0, n)
    pick_env = np.exp(-350.0 * t)
    pick = (pick_noise - 0.7 * np.roll(pick_noise, 1)) * pick_env * 0.45

    raw_signal = string_sound + pick

    # 4. Overdriven Tube Amplifier Stage (asymmetric soft-clipping)
    # Adding a slight DC offset creates natural 2nd-order tube warmth
    preamp_drive = drive * raw_signal + 0.1
    overdrive = np.tanh(preamp_drive) - np.tanh(0.1)

    # 5. Guitar Speaker Cabinet Emulation (rolls off harsh frequencies > 4.5 kHz)
    # A 7-point moving average filter removes digital buzz/fizz
    cab_kernel = np.hamming(9)
    cab_kernel /= np.sum(cab_kernel)
    cab_sound = np.convolve(overdrive, cab_kernel, mode='same')

    # 6. Note envelope (smooth note-off release to avoid pops)
    r_len = min(int(0.04 * sr), n // 4)
    if r_len > 0:
        release_env = np.ones(n)
        release_env[-r_len:] = np.linspace(1.0, 0.0, r_len)
        cab_sound *= release_env

    return amp * cab_sound

def render_guitar_melody(melody_events, amp=0.5, sr=SR):
    """
    Renders note sequence and passes it through an analog-style tape delay.
    """
    notes = [electric_guitar_note(f, d, amp, sr=sr) for f, d in melody_events]
    dry_track = np.concatenate(notes)

    # 7. Studio Delay / Echo effect (180ms delay with 35% feedback)
    delay_samples = int(0.18 * sr)
    wet_track = np.zeros(len(dry_track) + delay_samples * 2)
    wet_track[:len(dry_track)] = dry_track

    # First echo
    wet_track[delay_samples : delay_samples + len(dry_track)] += 0.30 * dry_track
    # Second echo
    wet_track[delay_samples * 2 : delay_samples * 2 + len(dry_track)] += 0.10 * dry_track

    # Peak normalization
    final_output = wet_track / np.max(np.abs(wet_track)) * 0.95
    return final_output

# ==============================================================================
# TWINKLE TWINKLE LITTLE STAR — ELECTRIC GUITAR LEAD
# ==============================================================================

# (MIDI note, duration in seconds)
# Lowered one octave (MIDI 48-60) for a warmer, throatier guitar register
melody = [
    (48, 0.5), (48, 0.5), (55, 0.5), (55, 0.5),
    (57, 0.5), (57, 0.5), (55, 1.0),
    (53, 0.5), (53, 0.5), (52, 0.5), (52, 0.5),
    (50, 0.5), (50, 0.5), (48, 1.0),
]

events = [(note_to_freq(n), d) for n, d in melody]
melody_audio = render_guitar_melody(events, amp=0.6)

print("🎸 Lead Electric Guitar — Twinkle Twinkle Little Star")
display(Audio(melody_audio, rate=SR))

🎸 Lead Electric Guitar — Twinkle Twinkle Little Star


Cell 4 — 🥁 Rhythm (pattern & placement of sounds in time)

In [51]:
# def kick(duration=0.2, sr=SR):
#     """Low sine sweep — sounds like a kick drum."""
#     t = np.linspace(0, duration, int(sr * duration), endpoint=False)
#     freq = np.linspace(150, 50, len(t))
#     return 0.9 * np.sin(2 * np.pi * freq * t) * np.exp(-8 * t)

# def snare(duration=0.15, sr=SR):
#     """Filtered noise burst — sounds like a snare/clap."""
#     t = np.linspace(0, duration, int(sr * duration), endpoint=False)
#     noise = np.random.randn(len(t))
#     return 0.4 * noise * np.exp(-20 * t)

# def hat(duration=0.05, sr=SR):
#     """Short high-frequency noise — hi-hat."""
#     t = np.linspace(0, duration, int(sr * duration), endpoint=False)
#     noise = np.random.randn(len(t))
#     return 0.25 * noise * np.exp(-60 * t)

# # 16-step pattern @ 120 BPM (each step = 0.125s => 8th notes at 120bpm~.25s)
# step_dur = 0.125
# steps = 16

# # Patterns: 1 = hit, 0 = rest
# kick_pat  = [1,0,0,0, 0,0,1,0, 0,0,1,0, 0,0,0,0]
# snare_pat = [0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0]
# hat_pat   = [1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]

# def render_pattern(pattern, sound_fn):
#     track = np.zeros(int(SR * step_dur * steps))
#     for i, hit in enumerate(pattern):
#         if hit:
#             start = int(i * step_dur * SR)
#             s = sound_fn()
#             end = min(start + len(s), len(track))
#             track[start:end] += s[:end - start]
#     return track

# beat = render_pattern(kick_pat, kick) \
#      + render_pattern(snare_pat, snare) \
#      + render_pattern(hat_pat, hat)

# beat = beat / np.max(np.abs(beat)) * 0.9
# print("🥁 Rhythm — basic 16-step drum pattern (kick / snare / hi-hat)")
# display(Audio(beat, rate=SR))

In [52]:
import numpy as np
from IPython.display import Audio, display

# Audio configuration
SR = 44100
BPM = 70                     # Standard tempo for Yearning Beat
step_dur = 60.0 / BPM / 4    # 16th-note step (~0.178s at 84 BPM)
steps = 64                   # 4 full measures (16 steps * 4 bars)

# ==============================================================================
# 1. INSTRUMENT SYNTHESIS (Tuned to the Yearning Beat sound)
# ==============================================================================

def kick_808(duration=0.85, sr=SR, base_freq=50.0, punch_freq=160.0):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    pitch_env = base_freq + punch_freq * np.exp(-55.0 * t)
    phase = 2.0 * np.pi * np.cumsum(pitch_env) / sr
    sub_body = np.sin(phase) * np.exp(-3.8 * t)
    click = np.random.randn(len(t)) * np.exp(-350.0 * t) * 0.12
    return np.tanh(1.8 * (sub_body + click)) * 0.95

def snare_yearning(duration=0.20, sr=SR):
    """
    Layered R&B/Pop snare:
    - 200 Hz body resonance for a clean chest-thump.
    - Bright, filtered wire crack with a tight tail.
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)

    # Fundamental body strike
    body = np.sin(2.0 * np.pi * 205.0 * t) * np.exp(-32.0 * t) * 0.45

    # Crisp wire crack
    noise = np.random.uniform(-1.0, 1.0, len(t))
    wires = (noise - 0.72 * np.roll(noise, 1)) * np.exp(-22.0 * t) * 0.65

    return np.tanh(1.5 * (body + wires)) * 0.85

def hat_yearning(duration=0.04, sr=SR):
    """Clean, high-frequency closed hi-hat."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    noise = np.random.uniform(-1.0, 1.0, len(t))
    sizzle = np.diff(np.diff(noise, prepend=[0, 0]))
    return sizzle * np.exp(-110.0 * t) * 0.32

# ==============================================================================
# 2. 4-BAR (64-STEP) TRANSCRIBED PATTERNS
# ==============================================================================

# Transcribed directly from the 4-measure waveform
# Bar 1 (0-15) | Bar 2 (16-31) | Bar 3 (32-47) | Bar 4 (48-63)

kick_pat = [
    # Bar 1: Intro groove
    1, 0, 1, 0,  0, 0, 0, 0,  1, 0, 1, 0,  0, 0, 1, 0,
    # Bar 2: Push on the 16ths
    1, 0, 0, 0,  0, 0, 1, 0,  0, 0, 1, 0,  0, 1, 0, 0,
    # Bar 3: Main pocket
    1, 0, 1, 0,  0, 0, 0, 0,  1, 0, 1, 0,  0, 0, 1, 0,
    # Bar 4: Groove leading into the fill[cite: 1]
    1, 0, 0, 0,  0, 0, 1, 0,  1, 0, 0, 0,  0, 0, 0, 0
]

snare_pat = [
    # Bar 1: Backbeat on 2 & 4
    0, 0, 0, 0,  1, 0, 0, 0,  0, 0, 0, 0,  1, 0, 0, 0,
    # Bar 2: Backbeat
    0, 0, 0, 0,  1, 0, 0, 0,  0, 0, 0, 0,  1, 0, 0, 0,
    # Bar 3: Backbeat
    0, 0, 0, 0,  1, 0, 0, 0,  0, 0, 0, 0,  1, 0, 0, 0,
    # Bar 4: Backbeat followed by the dense 6-hit fill on the far right[cite: 1]
    0, 0, 0, 0,  1, 0, 0, 0,  0, 0, 1, 1,  1, 1, 1, 1
]

# Steady 16th-note hat foundation across the groove
hat_pat = [1] * 56 + [0] * 8  # Drops out during the final fill to let the snare lead[cite: 1]

# ==============================================================================
# 3. SEQUENCING & STEREO RENDERING
# ==============================================================================

def render_pattern(pattern, sound_fn, tail_sec=0.5):
    total_len = int(SR * step_dur * steps + SR * tail_sec)
    track = np.zeros(total_len)

    for i, count in enumerate(pattern):
        if count > 0:
            sub_dur = step_dur / count
            for sub in range(count):
                start = int((i * step_dur + sub * sub_dur) * SR)
                sample = sound_fn()
                end = start + len(sample)
                track[start:end] += sample

    return track

# Render stem tracks
kick_track  = render_pattern(kick_pat, kick_808)
snare_track = render_pattern(snare_pat, snare_yearning)
hat_track   = render_pattern(hat_pat, hat_yearning) * 0.75

# Mix and peak normalize
beat = kick_track + snare_track + hat_track
beat = beat / np.max(np.abs(beat)) * 0.95

# Convert to stereo to match the original Logic / GarageBand audio region[cite: 1]
stereo_beat = np.vstack([beat, beat])

print(f"🥁 'Yearning Beat' Groove — 4 Measures @ {BPM} BPM")
display(Audio(stereo_beat, rate=SR))

🥁 'Yearning Beat' Groove — 4 Measures @ 70 BPM


Cell 5 — 🎧 Put it all together: Melody + Harmony + Rhythm

In [53]:
# Layer a melody on top of the drum beat with chords underneath.

# 1. Rhythm (length = 16 * 0.125 = 2.0s). Repeat 4x for ~8s
beat_loop = np.tile(beat, 1)

# 2. Harmony: I–vi–IV–V, each 2s, aligned with beat
chord_prog = [
    [60, 64, 67], [57, 60, 64], [53, 57, 60], [55, 59, 62]
]
harmony_audio = np.concatenate([
    mix([tone(note_to_freq(n), 2.0, amp=0.25) for n in chord])
    for chord in chord_prog
])

# 3. Melody on top (reuse Twinkle, played twice)
melody_loop = np.tile(melody_audio, 2)

# Align all to same length
L = max(len(beat_loop), len(harmony_audio), len(melody_loop))
def pad(x): return np.pad(x, (0, L - len(x)))

final = pad(beat_loop) * 0.8 + pad(harmony_audio) * 0.9 + pad(melody_loop) * 0.7
final = final / np.max(np.abs(final)) * 0.95

print("🎧 Full mix: rhythm + harmony + melody")
display(Audio(final, rate=SR))

🎧 Full mix: rhythm + harmony + melody
